# DHFR Antifolate Affinity — Project 1 (Colab runner)

Structure-aware **protein–ligand binding affinity** prediction. A two-branch model:
a **GNN over the ligand graph** (RDKit + PyTorch Geometric) + an **ESM-2 protein embedding**
of the DHFR target → predicted pIC50. Mirrors the ClusterFocusedHybrid two-branch design.

**Runtime:** set `Runtime → Change runtime type → GPU` (T4 is fine for the 35M ESM model).
Run the cells top to bottom.

## 1. Clone the repo and install
Replace the URL with your GitHub repo once you've pushed it.

In [ ]:
# If running in Colab, clone your repo. Locally, skip this cell.
import os, sys
REPO_URL = "https://github.com/<your-username>/dhfr-affinity.git"
if not os.path.exists("dhfr-affinity"):
    !git clone $REPO_URL
%cd dhfr-affinity

In [ ]:
# Install. torch is preinstalled on Colab; we add PyG + chem/bio deps.
!pip -q install rdkit torch_geometric fair-esm chembl_webresource_client
!pip -q install -e .
print("install done")

## 2. Add DHFR sequences
Edit `data/dhfr_sequences.fasta` with real wild-type sequences from UniProt
(E. coli folA = **P0ABQ4**, human DHFR = **P00374**), and add any resistant
variants as extra `>label` records. The cell below fetches them automatically.

In [ ]:
# Auto-fetch DHFR sequences from UniProt into the FASTA file.
import urllib.request, os
UNIPROT = {"e_coli":"P0ABQ4", "human":"P00374"}   # add variants manually
os.makedirs("data", exist_ok=True)
lines = []
for label, acc in UNIPROT.items():
    url = f"https://rest.uniprot.org/uniprotkb/{acc}.fasta"
    fasta = urllib.request.urlopen(url).read().decode()
    seq = "".join(fasta.splitlines()[1:])
    lines.append(f">{label}\n{seq}")
open("data/dhfr_sequences.fasta","w").write("\n".join(lines)+"\n")
print(open("data/dhfr_sequences.fasta").read()[:300])

## 3. Pull + clean ChEMBL data
First run hits the network; results are cached to `outputs/dhfr_clean.csv`. Use `max_per_target` to keep the first run quick.

In [ ]:
from dhfr_affinity.data import fetch_dhfr_bioactivities, clean_bioactivities
import os, pandas as pd
os.makedirs("outputs", exist_ok=True)
cache = "outputs/dhfr_clean.csv"
if os.path.exists(cache):
    df = pd.read_csv(cache)
else:
    raw = fetch_dhfr_bioactivities(max_per_target=1500)   # raise/remove later
    df = clean_bioactivities(raw)
    df.to_csv(cache, index=False)
print(df.shape); df.head()

In [ ]:
df["pic50"].hist(bins=40); 
import matplotlib.pyplot as plt; plt.xlabel("pIC50"); plt.title("label distribution"); plt.show()
df.groupby("organism").size()

## 4. Split (scaffold), embed proteins, build datasets

In [ ]:
from dhfr_affinity.data import scaffold_split
from dhfr_affinity.dataset import build_dataset, load_sequences
from dhfr_affinity.features.protein import ESMEmbedder
from dhfr_affinity.utils import set_seed, get_device
set_seed(42); device = get_device(); print("device", device)

parts = scaffold_split(df, frac_train=0.8, frac_valid=0.1, seed=42)
sequences = load_sequences("data/dhfr_sequences.fasta")
embedder = ESMEmbedder("esm2_t12_35M_UR50D", "data/esm_cache", str(device))
splits = {name: build_dataset(p, sequences, embedder) for name, p in parts.items()}
protein_dim = splits["train"][0].protein.shape[1]; print("protein_dim", protein_dim)

## 5. Train ligand-only baseline vs. two-branch hybrid
The honesty check: does adding the protein branch actually beat ligand-only?

In [ ]:
from dhfr_affinity.train import make_loaders, train_model, evaluate_split
from dhfr_affinity.models import LigandOnlyModel, TwoBranchAffinityModel
from dhfr_affinity.evaluate import comparison_table, plot_predictions, plot_history
import matplotlib.pyplot as plt

loaders = make_loaders(splits, batch_size=64)
cfg = dict(gnn_hidden=128, gnn_layers=3, ligand_out=128, protein_out=128, head_hidden=256, dropout=0.1)
results, histories = {}, {}
for name, ctor in [("ligand_only", LigandOnlyModel), ("two_branch", TwoBranchAffinityModel)]:
    print("==", name, "==")
    model = ctor(protein_dim=protein_dim, **cfg)
    model, hist = train_model(model, loaders, device, epochs=100, patience=15)
    results[name] = evaluate_split(model, loaders["test"], device)
    histories[name] = hist
    globals()[name+"_model"] = model

comparison_table(results).round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11,5))
for ax,(name,res) in zip(axes, results.items()):
    plot_predictions(res["y_true"], res["y_pred"], title=f"{name} (test)", ax=ax)
plt.tight_layout(); plt.show()

## 6. Interpretability — which atoms drive the prediction

In [ ]:
from dhfr_affinity.interpret import atom_attribution, draw_attribution
mol = splits["test"][0]
sal = atom_attribution(two_branch_model, mol, device)
print("SMILES:", mol.smiles)
draw_attribution(mol.smiles, sal)

## 7. E. coli case study
Slice the test predictions to E. coli only — the headline tie-back to the TMP-SMX
resistance story. (Most affinity data is human DHFR; E. coli is the case study.)

In [ ]:
import numpy as np
ecoli = [(d.smiles, float(d.y)) for d in splits["test"] if d.organism=="e_coli"]
print(f"{len(ecoli)} E. coli test compounds")
# (extend: predict on each, compare WT vs a resistant DHFR variant embedding)